<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°06


**Objetivo**: Aplicar técnicas básicas de **Machine Learning**, desde la preparación de datos hasta el entrenamiento y evaluación de modelos.




<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/ec/Anscombe%27s_quartet_3.svg/1200px-Anscombe%27s_quartet_3.svg.png" width="500"/>
</p>

El **cuarteto de Anscombe** es un ejemplo clásico en estadística que ilustra cómo diferentes conjuntos de datos pueden compartir las mismas propiedades estadísticas, como media, varianza y correlación, pero presentan comportamientos muy distintos cuando se visualizan gráficamente. Cada uno de los cuatro conjuntos consiste en once puntos (x, y) y fue creado por el estadístico F. J. Anscombe en 1973. Esta herramienta resalta la importancia de la visualización de datos para evitar interpretaciones erróneas basadas únicamente en análisis numéricos.

**Descripción del conjunto**

1. **Propiedades estadísticas comunes:** Todos los conjuntos tienen el mismo valor promedio para las variables \(x\) e \(y\), la misma varianza para \(x\) e \(y\), y una correlación lineal idéntica.
2. **Diferencias gráficas:** A pesar de sus similitudes estadísticas, los cuatro conjuntos presentan gráficos muy distintos:
   - El primer conjunto muestra una relación lineal simple.
   - El segundo conjunto tiene una relación no lineal, con una curva clara.
   - El tercer conjunto tiene una relación lineal clara, pero con un punto atípico que influye significativamente.
   - El cuarto conjunto tiene la mayoría de los puntos alineados verticalmente, con un punto atípico que afecta la correlación.

Este cuarteto enfatiza que las estadísticas descriptivas por sí solas pueden no capturar la esencia completa de los datos, subrayando la necesidad de utilizar visualizaciones en cualquier análisis exploratorio de datos.

In [ ]:
# Importar las bibliotecas necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuración de los gráficos
%matplotlib inline
sns.set_theme(style="whitegrid")  # Establece un tema general para los gráficos
sns.set_palette("deep", desat=0.6)
plt.rcParams['figure.figsize'] = (12, 8)  # Ajuste del tamaño de las figuras

# Cargar los datos del cuarteto de Anscombe
data = sns.load_dataset("anscombe")

# Mostrar las primeras filas del conjunto de datos
data.head()

,dataset,x,y
0,I,10.0,8.04
1,I,8.0,6.95
2,I,13.0,7.58
3,I,9.0,8.81
4,I,11.0,8.33


Con base en la información presentada y el análisis realizado, les invitamos a reflexionar y responder las siguientes preguntas. Estas preguntas están diseñadas para profundizar en su comprensión del cuarteto de Anscombe y fomentar un análisis crítico de los datos:



1. Cree un gráfico de dispersión (scatter plot) para cada uno de los cuatro grupos del cuarteto de Anscombe. A partir de la visualización, ¿puede identificar diferencias significativas entre los grupos? ¿Qué características particulares observa en cada uno que sugieren comportamientos distintos?



In [ ]:
# FIXME

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, dataset_name in enumerate(data['dataset'].unique()):
    subset = data[data['dataset'] == dataset_name]
    sns.scatterplot(x='x', y='y', data=subset, ax=axes[i])
    sns.regplot(x='x', y='y', data=subset, ax=axes[i], scatter=False, color='red', line_kws={'linestyle':'--'})
    axes[i].set_title(f'Conjunto de Datos: {dataset_name}', fontsize=14)
    axes[i].set_xlabel('X', fontsize=12)
    axes[i].set_ylabel('Y', fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.7)

plt.suptitle('Gráficos de Dispersión para el Cuarteto de Anscombe', fontsize=18, y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


### Análisis de los Gráficos de Dispersión:

Observando los gráficos de dispersión generados para cada conjunto de datos del cuarteto de Anscombe, se pueden identificar las siguientes diferencias significativas y características particulares:

*   **Conjunto I (Superior Izquierda):** Este gráfico muestra una clara relación lineal positiva. Los puntos están distribuidos de manera uniforme alrededor de una línea recta, lo que sugiere que un modelo de regresión lineal sería apropiado para este conjunto de datos.

*   **Conjunto II (Superior Derecha):** Aquí, la relación entre `x` y `y` es claramente no lineal, exhibiendo una forma parabólica o curvilínea. Aunque hay una tendencia general de aumento, la forma no es recta, lo que indica que un modelo lineal simple no capturaría adecuadamente la relación subyacente. La línea de regresión lineal (punteada en rojo) muestra una tendencia general, pero no se ajusta bien a la curva de los puntos.

*   **Conjunto III (Inferior Izquierda):** Este conjunto también presenta una relación predominantemente lineal. Sin embargo, hay un `outlier` prominente en la parte superior derecha que tira de la línea de regresión hacia arriba. Si se ignorara este punto, la relación lineal sería más fuerte y con una pendiente diferente. La presencia de este valor atípico distorsiona la percepción de la relación lineal.

*   **Conjunto IV (Inferior Derecha):** Este gráfico es muy distinto. La mayoría de los puntos de datos tienen el mismo valor de `x` (alrededor de 8), y solo hay un punto con un valor de `x` muy diferente (alrededor de 19). Este único punto atípico ejerce una influencia enorme en la línea de regresión lineal, forzándola a tener una pendiente. Sin ese punto, no habría una relación lineal discernible, ya que todos los demás puntos formarían una línea vertical. La línea de regresión lineal parece tener una pendiente positiva, pero es casi enteramente dictada por el único punto alejado.

Estas visualizaciones son cruciales porque, como veremos en los siguientes pasos, las estadísticas descriptivas para todos estos conjuntos serán muy similares, pero sus patrones subyacentes son fundamentalmente diferentes. Esto subraya la importancia de la visualización de datos antes de cualquier análisis estadístico o modelado.

2. Utilice el comando `describe` para generar un resumen de las medidas estadísticas más relevantes para cada uno de los grupos del cuarteto de Anscombe. A partir de estos resultados, interprete las estadísticas obtenidas, destacando las características más significativas de cada grupo y cómo pueden influir en la comprensión de sus respectivas distribuciones.


In [ ]:
# FIXME

3. Ajuste un modelo de regresión lineal para cada grupo utilizando **sklearn**. Calcule las métricas de evaluación, como el error cuadrático medio (MSE) y R², y grafique los resultados de la regresión. Interprete los resultados y su impacto en la calidad del ajuste.



In [ ]:
# FIXME

4. Es evidente que el ajuste lineal no es adecuado para algunos grupos. Existen diversas estrategias para abordar este problema, como eliminar outliers o emplear diferentes modelos de regresión. Identifique una estrategia que podría mejorar el ajuste del modelo de regresión lineal y, si lo considera necesario, implemente otros modelos alternativos para aquellos casos donde el ajuste lineal resulte inadecuado.

In [ ]:
# FIXME